# Week 52, Day 5: Pruning

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w052d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def count_nodes(numbers, target, prune):
    numbers = sorted(numbers, reverse=True)
    suffix = [0] * (len(numbers) + 1)
    for i in range(len(numbers) - 1, -1, -1):
        suffix[i] = suffix[i + 1] + numbers[i]
    visited = [0]
    def search(i, total):
        visited[0] += 1
        if total == target:
            return True
        if i == len(numbers) or total > target:
            return False
        if prune and total + suffix[i] < target:
            return False                       # even taking everything left falls short
        return search(i + 1, total + numbers[i]) or search(i + 1, total)
    found = search(0, 0)
    return found, visited[0]

nums = [41, 34, 33, 29, 27, 21, 19, 13, 9, 7, 5, 3]
print(count_nodes(nums, 240, prune=False))
print(count_nodes(nums, 240, prune=True))

In [ ]:
def subsets_summing(numbers, target):
    numbers = sorted(numbers)
    suffix = [0] * (len(numbers) + 1)
    for i in range(len(numbers) - 1, -1, -1):
        suffix[i] = suffix[i + 1] + numbers[i]
    out, current = [], []
    def search(i, total):
        if total == target:
            out.append(list(current))
            return
        if i == len(numbers) or total + numbers[i] > target or total + suffix[i] < target:
            return                     # sorted: if the smallest left overshoots, all do
        current.append(numbers[i])
        search(i + 1, total + numbers[i])
        current.pop()
        search(i + 1, total)
    search(0, 0)
    return out

print(subsets_summing([3, 34, 4, 12, 5, 2], 9))

In [ ]:
def knapsack_bb(items, capacity):
    """items: (value, weight). Branch and bound with the fractional bound."""
    items = sorted(items, key=lambda it: it[0] / it[1], reverse=True)
    best, visited = [0], [0]

    def bound(i, value, room):
        for v, w in items[i:]:
            if w <= room:
                value, room = value + v, room - w
            else:
                return value + v * room / w          # a fraction of the next item
        return value

    def search(i, value, room):
        visited[0] += 1
        best[0] = max(best[0], value)
        if i == len(items) or bound(i, value, room) <= best[0]:
            return                                    # can't beat what we have
        v, w = items[i]
        if w <= room:
            search(i + 1, value + v, room - w)        # take it first: good answers early
        search(i + 1, value, room)

    search(0, 0, capacity)
    return best[0], visited[0]

import random
rng = random.Random(1)
things = [(rng.randint(10, 100), rng.randint(5, 50)) for _ in range(30)]
print(knapsack_bb(things, 200), "out of a possible", 2 ** 31 - 1, "nodes")

In [ ]:
def can_split(text, words):
    words = set(words)
    failed, visited = set(), [0]
    def search(i):
        visited[0] += 1
        if i == len(text):
            return True
        if i in failed:
            return False
        for j in range(i + 1, len(text) + 1):
            if text[i:j] in words and search(j):
                return True
        failed.add(i)
        return False
    return search(0), visited[0]

print(can_split("a" * 25 + "b", ["a", "aa", "aaa", "aaaa"]))

In [ ]:
def fewest_steps(start, goal, max_depth=20):
    """Reach goal from start using steps x + 1, x * 2 or x * 3."""
    def search(x, depth, path):
        if x == goal:
            return list(path)
        if depth == 0 or x > goal:
            return None
        for nxt in (x * 3, x * 2, x + 1):
            path.append(nxt)
            found = search(nxt, depth - 1, path)
            if found:
                return found
            path.pop()
        return None
    for limit in range(max_depth + 1):
        found = search(start, limit, [start])
        if found:
            return found
    return None

print(fewest_steps(1, 100))

> **Key idea.** Pruning skips branches that provably can't contain what you want. Feasibility pruning asks whether success is still possible (suffix sums make it O(1)); every rule must be safe, never discarding a real solution. Branch and bound keeps the best answer so far and prunes any branch whose optimistic bound can't beat it; for 0/1 knapsack, the fractional knapsack of what's left is that bound. Try promising choices first so good answers appear early, break symmetry so equivalent solutions aren't all searched, and remember failed states (the whole state, not part of it) to avoid repeating dead ends. Iterative deepening finds the shallowest solution with depth-first memory. Measure each idea by counting visited nodes.

### Exercise: Pruned subset sums

Write `count_subsets(numbers, target)`: the number of subsets of positive whole numbers summing exactly to target, with two prunings (sort ascending; stop when the smallest remaining number overshoots, and when the total plus all remaining numbers falls short, using suffix sums). Return `(count, nodes_visited)`. The checks compare with brute force and require far fewer nodes than the 2ⁿ⁺¹ − 1 of an unpruned search on a large example.

In [ ]:
def count_subsets(numbers, target):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef test_subsets():\n    \"pruned subsets\"\n    assert count_subsets([3, 34, 4, 12, 5, 2], 9)[0] == 2 and count_subsets([], 0)[0] == 1 and count_subsets([5], 3)[0] == 0\n    rng = random.Random(1)\n    for _ in range(100):\n        nums = [rng.randint(1, 10) for _ in range(rng.randint(0, 9))]\n        t = rng.randint(0, 25)\n        want = sum(1 for r in range(len(nums) + 1) for c in itertools.combinations(nums, r) if sum(c) == t)\n        assert count_subsets(nums, t)[0] == want\n    big = [rng.randint(10, 60) for _ in range(24)]\n    count, nodes = count_subsets(big, 150)\n    assert nodes < 2 ** 25 // 50, nodes")

### Exercise: Branch and bound

Write `knapsack(items, capacity)` for whole items `(value, weight)` with positive weights: `(best_value, nodes_visited)` by branch and bound with the fractional bound, items in value-per-weight order, taking each item before skipping it. Then `chosen(items, capacity)`: the indices (in the original list) of one best choice, sorted. The checks compare with a dynamic-programming answer and require far fewer nodes than 2ⁿ on 40 items.

In [ ]:
def knapsack(items, capacity):
    ___

def chosen(items, capacity):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef dp(items, cap):\n    best = [0] * (cap + 1)\n    for v, w in items:\n        for c in range(cap, w - 1, -1):\n            best[c] = max(best[c], best[c - w] + v)\n    return best[cap]\ndef test_knapsack():\n    \"knapsack\"\n    assert knapsack([(6, 6), (5, 5), (5, 5)], 10)[0] == 10 and knapsack([], 5)[0] == 0\n    rng = random.Random(2)\n    for _ in range(60):\n        items = [(rng.randint(1, 30), rng.randint(1, 15)) for _ in range(rng.randint(0, 12))]\n        cap = rng.randint(0, 40)\n        assert knapsack(items, cap)[0] == dp(items, cap)\n        idx = chosen(items, cap)\n        assert sum(items[i][1] for i in idx) <= cap and sum(items[i][0] for i in idx) == dp(items, cap)\n    big = [(rng.randint(10, 100), rng.randint(5, 50)) for _ in range(40)]\n    value, nodes = knapsack(big, 300)\n    assert value == dp(big, 300) and nodes < 200_000, nodes")

### Exercise: The shortest tour

A delivery round must visit every town once and return to the start (town 0); `dist[i][j]` is the distance. Write `shortest_tour(dist)`: `(length, tour)` with the tour as a list starting and ending at 0, by branch and bound: extend partial tours, and prune when the length so far plus the **cheapest edge out of every unvisited town** (plus the cheapest way back to 0) can't beat the best tour found. Seed the best with the greedy nearest-neighbour tour, and try nearer towns first. Then `tour_length(dist, tour)`.

In [ ]:
def shortest_tour(dist):
    ___

def tour_length(dist, tour):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random, time\ndef brute(d):\n    n = len(d)\n    best = None\n    for p in itertools.permutations(range(1, n)):\n        t = (0,) + p + (0,)\n        L = sum(d[a][b] for a, b in zip(t, t[1:]))\n        best = L if best is None else min(best, L)\n    return best if n > 1 else 0\ndef test_tsp():\n    \"tours\"\n    d = [[0, 2, 9, 10], [1, 0, 6, 4], [15, 7, 0, 8], [6, 3, 12, 0]]\n    length, tour = shortest_tour(d)\n    assert length == 21 and tour[0] == tour[-1] == 0 and tour_length(d, tour) == 21 and sorted(tour[:-1]) == [0, 1, 2, 3]\n    assert shortest_tour([[0]]) == (0, [0, 0])\n    rng = random.Random(3)\n    for _ in range(20):\n        n = rng.randint(2, 7)\n        d = [[0 if i == j else rng.randint(1, 30) for j in range(n)] for i in range(n)]\n        length, tour = shortest_tour(d)\n        assert length == brute(d) and tour_length(d, tour) == length\n    pts = [(rng.randint(0, 100), rng.randint(0, 100)) for _ in range(11)]\n    d = [[abs(a[0] - b[0]) + abs(a[1] - b[1]) for b in pts] for a in pts]\n    t0 = time.perf_counter()\n    shortest_tour(d)\n    assert time.perf_counter() - t0 < 8")

### Exercise: Half the board

Write `count_queens_symmetric(n)`: the number of N-queens solutions, computed by searching only boards whose first-row queen is in the **left half** of the board, doubling that, and (for odd n) adding the solutions with the first queen in the middle column. Use any correct queen search for the pieces. Then `nodes_saved(n)`: the fraction of first-row choices the symmetric search avoids, `(n // 2) / n` rounded to 3 places.

In [ ]:
def count_queens_symmetric(n):
    ___

def nodes_saved(n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_mirror():\n    \"symmetry\"\n    assert [count_queens_symmetric(n) for n in range(1, 10)] == [1, 0, 0, 2, 10, 4, 40, 92, 352]\n    assert nodes_saved(8) == 0.5 and nodes_saved(5) == 0.4")

### Exercise: Iterative deepening

Starting from a number, the allowed steps are `x + 1`, `x * 2` and `x * 3`. Write `fewest_steps(start, goal)`: the shortest list of numbers from start to goal (inclusive), by iterative deepening with a depth-first search that tries `x * 3`, then `x * 2`, then `x + 1`, and prunes when x exceeds the goal. Then `depth_needed(start, goal)`: the length of that list minus one.

In [ ]:
def fewest_steps(start, goal):
    ___

def depth_needed(start, goal):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from collections import deque\ndef bfs(s, g):\n    d = {s: 0}\n    q = deque([s])\n    while q:\n        x = q.popleft()\n        if x == g:\n            return d[x]\n        for y in (x + 1, x * 2, x * 3):\n            if y <= g and y not in d:\n                d[y] = d[x] + 1\n                q.append(y)\ndef test_deepen():\n    \"deepening\"\n    path = fewest_steps(1, 100)\n    assert path[0] == 1 and path[-1] == 100 and len(path) - 1 == bfs(1, 100)\n    assert all(b in (a + 1, a * 2, a * 3) for a, b in zip(path, path[1:]))\n    assert fewest_steps(7, 7) == [7] and depth_needed(1, 100) == bfs(1, 100)\n    for g in (17, 55, 81, 127):\n        assert depth_needed(2, g) == bfs(2, g)")

### Bug hunt: A prune that's too eager

This subset-sum search prunes when the total plus everything remaining is **at most** the target, but "exactly equal" is a success, not a dead end: it throws away the solution that needs every remaining number and prints `False`. Prune only when it's strictly less, so it prints `True`.

In [ ]:
numbers = [5, 4, 3]
target = 12
suffix = [12, 7, 3, 0]

def search(i, total):
    if total == target:
        return True
    if i == len(numbers) or total + suffix[i] <= target:
        return False
    return search(i + 1, total + numbers[i]) or search(i + 1, total)

print(search(0, 0))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"safe pruning\"\n    expect_output(\"True\")")

### Bug hunt: A pessimistic bound

This knapsack's "bound" is just the value taken so far, which is **not** optimistic: it says no branch can improve on the best, so the search stops almost at once with a poor answer, printing `6`. Use the fractional bound over the remaining items so it prints `10`.

In [ ]:
items = sorted([(6, 6), (5, 5), (5, 5)], key=lambda it: it[0] / it[1], reverse=True)
best = [0]

def search(i, value, room):
    best[0] = max(best[0], value)
    if i == len(items) or value <= best[0] and i > 0:
        return
    v, w = items[i]
    if w <= room:
        search(i + 1, value + v, room - w)
    search(i + 1, value, room)

search(0, 0, 10)
print(best[0])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"an optimistic bound\"\n    expect_output(\"10\")")

### Bug hunt: Remembering too little

This search asks whether the text splits into dictionary words using **at most** `k` words, and remembers failures. But it remembers only the position, and the state is really the position **and** the words still allowed: reaching position 2 with one word left fails (`"b"` leaves a `"b"` over), and that failure wrongly blocks reaching position 2 later with two words to spare. It prints `False`. Remember `(i, left)` pairs so it prints `True`.

In [ ]:
words = {"a", "aa", "b"}
text = "aabb"
failed = set()

def search(i, left):
    if i == len(text):
        return True
    if left == 0 or i in failed:
        return False
    for j in range(i + 1, len(text) + 1):
        if text[i:j] in words and search(j, left - 1):
            return True
    failed.add(i)
    return False

print(search(0, 3))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the whole state\"\n    expect_output(\"True\")")

### Puzzle: Branch and bound

Arrange the lines into the branch-and-bound search (a `bound` function is given). One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the suffix sums of [5, 3, 2], from index 0 to 3, as a list
q = ___   # the fractional-knapsack bound for items (10, 5), (6, 6) with room 8 (best ratio first)
r = ___   # True or False: an optimistic bound may overestimate what a branch can achieve
s = ___   # how many depth limits iterative deepening tries before finding a solution 3 steps away
t = ___   # True or False: a prune that sometimes discards a real solution only makes the search slower

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [10, 5, 2, 0] and q == 13.0 and r is True, \"p, q and r\"\n    assert s == 4 and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: packing bins

Items of given sizes must be packed into the **fewest** bins of a fixed capacity (lorries, disks, shelves). Write `fewest_bins(sizes, capacity)` by branch and bound:

- Start the best answer from the **first-fit decreasing** heuristic: sort sizes largest first and put each in the first bin it fits (a new bin if none). Write it as `first_fit_decreasing(sizes, capacity)`, returning the number of bins.
- Search: place items largest first, each into an existing bin it fits or one new bin. Prune when the bins in use are already at least the best; use the **lower bound** ⌈total size ÷ capacity⌉ to stop as soon as an answer reaches it; and, as in Wednesday's equal shares, skip a bin whose current load equals one already tried for this item.

Raise `ValueError` if any item is bigger than the capacity. Return the number of bins. The checks compare with brute force and include cases where first-fit decreasing isn't optimal.

In [ ]:
import math

def first_fit_decreasing(sizes, capacity):
    ___

def fewest_bins(sizes, capacity):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random, time\ndef brute(sizes, cap):\n    n = len(sizes)\n    for k in range(1, n + 1):\n        for labels in itertools.product(range(k), repeat=n):\n            loads = [0] * k\n            for s, l in zip(sizes, labels):\n                loads[l] += s\n            if max(loads) <= cap:\n                return k\n    return 0\ndef test_bins():\n    \"bins\"\n    assert fewest_bins([4, 8, 1, 4, 2, 1], 10) == 2 and fewest_bins([], 10) == 0\n    tricky = [6, 6, 5, 5, 4, 4, 3, 3, 2, 2]\n    assert first_fit_decreasing(tricky, 13) >= fewest_bins(tricky, 13)\n    try:\n        fewest_bins([11], 10)\n        assert False\n    except ValueError:\n        pass\n    rng = random.Random(4)\n    for _ in range(40):\n        sizes = [rng.randint(1, 9) for _ in range(rng.randint(1, 7))]\n        assert fewest_bins(sizes, 10) == brute(sizes, 10)\n    found = False\n    for _ in range(300):\n        sizes = [rng.randint(2, 8) for _ in range(8)]\n        if first_fit_decreasing(sizes, 10) > fewest_bins(sizes, 10):\n            found = True\n            break\n    assert found, \"first-fit decreasing isn't always optimal\"\ndef test_speed():\n    \"pruning\"\n    rng = random.Random(5)\n    sizes = [rng.randint(10, 60) for _ in range(22)]\n    t0 = time.perf_counter()\n    fewest_bins(sizes, 100)\n    assert time.perf_counter() - t0 < 8")

## Warm-up

Same answer, very different work. The pruned search notices, at every step, when the numbers still available can't possibly reach the target, and abandons the branch.

## Can it still succeed?

The simplest pruning asks: **is success still possible from here?** It needs a quick test that never says "no" wrongly (it may say "maybe" when the answer is really no; that only costs time). For sums:

- if the running total already exceeds the target (with positive numbers), stop;
- if the total plus **all** the remaining numbers can't reach the target, stop. Precompute **suffix sums** so this test is O(1).

The test must be **safe**: a prune that sometimes throws away a real solution makes the search wrong, not just slow. Write each pruning rule as "if this holds, no solution exists in this branch", and convince yourself it's true.

## Branch and bound

For **optimisation** ("the best", not "any"), keep the **best answer found so far**, and give each branch an **optimistic bound**: a value at least as good as anything in that branch. If the bound can't beat the best so far, prune. Better bounds prune more; a bound that's too optimistic still works but prunes less, and one that's too pessimistic is **wrong**.

The 0/1 knapsack (Monday's failure of greed) is the classic. Optimistic bound for a branch: the value taken so far plus the **fractional** knapsack of the remaining items into the remaining capacity (Monday's greedy, which can only do better than whole items). Sort items by value per weight once, so the bound is a quick scan.

This is **branch and bound**, introduced by Ailsa Land and Alison Doig in 1960 for integer programming, and still the core of the solvers used for scheduling and routing.

## Good choices first

The order in which choices are tried matters twice over. For **finding one** solution, trying likely choices first finds it sooner. For **branch and bound**, a good answer found early makes the bound prune everything weaker from then on. So:

- try the choice the greedy rule would make first (the knapsack above takes each item before skipping it, in value-per-weight order);
- for sudoku, fill the most constrained cell first (yesterday) and try its candidates in a sensible order;
- start with a decent answer from a quick heuristic (a greedy solution), so pruning starts strong from the first node.

## Symmetry and dead ends

**Symmetry**: if the problem looks the same after a reflection or rotation, solutions come in families, and searching every member is wasted work. N-queens is mirror-symmetric left to right, so count solutions whose first queen is in the **left half** and double them (taking care with the middle column on odd boards). Equal-share partitioning (Wednesday) skipped groups with the same current sum for the same reason: the groups are interchangeable.

**Dead ends**: if the same partial state can be reached by different routes, and it failed once, it will fail again. **Remember the states that failed** in a set and skip them. For example, "can this string be split into dictionary words?": once you know the suffix from position i can't be split, every route that reaches position i again can stop at once. This is memoisation (Week 16), applied to failures, and it's the bridge to next week's dynamic programming.

Without the `failed` set, that search would try every way to split 25 a's (about 7.5 million) before giving up.

## Iterative deepening

Depth-first search uses little memory but may wander deep down a long path when a short solution exists; breadth-first finds the shallowest solution but stores a whole level of states, which can be enormous. **Iterative deepening** combines them: run a depth-first search limited to depth 1, then 2, then 3, ..., until a solution appears. The first one found is the shallowest, and memory is only one path. Repeating the shallow levels costs surprisingly little, because most of the work is always in the deepest level. Richard Korf analysed it in 1985 and combined it with A\* bounds as IDA\*.

## Practice

## Quiz

## Daily challenge

## Before you go

Pick a puzzle you enjoy (a crossword, a jigsaw, a card game) and name one pruning rule you use without thinking: a reason you never even try certain moves. Tomorrow is the Forge: an exam timetable built with greedy choices and backtracking.